# **UNIVERSIDADE FEDERAL DO CEARÁ**
---
Disciplina: Introdução à análise em Big Data

---

Professor: Luiz Alexandre

---

Alunos:
1.   Júlio César Gama Feitosa Freitas - 583956
2.   Vitória Freire Rocha Teixeira de Oliveira - 587661

---
Data: 13/09/2026

# 🧪 Lab 2 — Sqoop: importar clientes e transações

## 🎯 Objetivo

Trazer os dados de um banco MySQL para o HDFS (rota cluster) ou reproduzir a mesma lógica de "ingestão de banco relacional" localmente com SQLite (rota sem admin).

> **Importante:** Como o ambiente utilizado é o Google Colab, não houve execução real de Sqoop, YARN ou HDFS distribuído; a atividade reproduz localmente a lógica de ingestão e particionamento.

## Configuração inicial

In [1]:
# Importação das bibliotecas e verificação dos arquivos de dados necessários
#from google.colab import files # Permite o upload de arquivos CSV no Colab
import os, sqlite3, shutil, pandas as pd

#uploaded = files.upload()

required = [
    "../customers_synthetic.csv",
    "../transactions_synthetic.csv",
    "../fraud_labels.csv"
]

missing = [f for f in required if not os.path.exists(f)] # Verifica se todos os arquivos necessários foram carregados
if missing:
    raise FileNotFoundError("Arquivos ausentes: " + ", ".join(missing))

print("\n✓ Os 3 datasets foram encontrados.")


✓ Os 3 datasets foram encontrados.


In [2]:
# Criação da estrutura de diretórios para o projeto
BASE = "../content/bigdata/"

for d in [
    "raw/customers",
    "raw/transactions",
    "raw/fraud_labels",
    "bronze",
    "silver",
    "gold"
]:
    os.makedirs(os.path.join(BASE, d), exist_ok=True)

print("✓ Estrutura preparada em /content/bigdata")

✓ Estrutura preparada em /content/bigdata


## Passo 1 — Criar o banco SQLite a partir dos CSVs

---



In [3]:
# Criação do banco SQLite a partir dos CSVs
DB_PATH = "/content/bigdata_course.db" # Define o caminho para o arquivo do banco de dados SQLite

if os.path.exists(DB_PATH):
    os.remove(DB_PATH) # Remove o arquivo do banco de dados se já existir para iniciar limpo

conn = sqlite3.connect(DB_PATH)

customers = pd.read_csv("../customers_synthetic.csv") # Carrega os CSVs de clientes
transactions = pd.read_csv("../transactions_synthetic.csv") # Carrega os CSVs de transações

customers.to_sql("customers", conn, if_exists="replace", index=False) # Salva o DataFrame como tabela no SQLite
transactions.to_sql("transactions", conn, if_exists="replace", index=False) # Salva o DataFrame como tabela no SQLite

conn.close()

print("✓ Banco SQLite criado")
print("✓ Tabela customers criada")
print("✓ Tabela transactions criada")

✓ Banco SQLite criado
✓ Tabela customers criada
✓ Tabela transactions criada


## Passo 2 — Conferir a carga

In [4]:
# Conferência da contagem de registros nas tabelas do banco de dados
conn = sqlite3.connect(DB_PATH) # Conecta ao banco de dados SQLite

n_customers = conn.execute(
    "SELECT COUNT(*) FROM customers" # Obtém a contagem de clientes
).fetchone()[0]

n_transactions = conn.execute(
    "SELECT COUNT(*) FROM transactions" # Obtém a contagem de transações
).fetchone()[0]

print(f"clientes:   {n_customers:,}") # Exibe as contagens formatadas
print(f"transações: {n_transactions:,}") # Exibe as contagens formatadas

conn.close()

clientes:   9,993
transações: 100,000


In [5]:
# Verificação da distribuição de transações fraudulentas
conn = sqlite3.connect(DB_PATH)

fraud = pd.read_sql_query(
    "SELECT is_fraud, COUNT(*) AS quantidade "
    "FROM transactions GROUP BY is_fraud", # Consulta a contagem de transações fraudulentas e não fraudulentas
    conn
)

display(fraud)
conn.close()

,is_fraud,quantidade
0,0,98167
1,1,1833


## Passo 3 — "Importar" para a camada raw (equivalente ao Sqoop)

Na Rota B, o lab usa SQLite para simular o banco de origem e Python/Pandas para exportar os dados para `bigdata/raw/`.


In [6]:
# Exportação do SQLite para a camada raw
conn = sqlite3.connect(DB_PATH)

customers_db = pd.read_sql("SELECT * FROM customers", conn) # Lê os dados da tabela customers do SQLite
transactions_db = pd.read_sql("SELECT * FROM transactions", conn) # Lê os dados da tabela transactions do SQLite

conn.close()

customers_raw = os.path.join(
    BASE, "raw/customers/customers_from_db.csv" # Define o caminho de saída para clientes
)

transactions_raw = os.path.join(
    BASE, "raw/transactions/transactions_from_db.csv" # Define o caminho de saída para transações
)

customers_db.to_csv(customers_raw, index=False) # Salva o DataFrame de clientes como CSV
transactions_db.to_csv(transactions_raw, index=False) # Salva o DataFrame de transações como CSV

print("✓ customers_from_db.csv exportado")
print("✓ transactions_from_db.csv exportado")

✓ customers_from_db.csv exportado
✓ transactions_from_db.csv exportado


In [7]:
# Validação da quantidade de registros após a exportação
c_check = pd.read_csv(customers_raw)
t_check = pd.read_csv(transactions_raw)

print(f"Clientes:    {len(c_check):,}")
print(f"Transações:  {len(t_check):,}")

assert len(c_check) == n_customers # Compara a quantidade de registros de clientes para garantir a integridade
assert len(t_check) == n_transactions # Compara a quantidade de registros de transações para garantir a integridade

print("\n✓ Quantidades preservadas.")

Clientes:    9,993
Transações:  100,000

✓ Quantidades preservadas.


## Passo 4 — Simular paralelismo (o que o `-m 4` do Sqoop faz)

O lab original divide as transações em quatro partes para representar os quatro mappers do Sqoop. Espera-se aproximadamente 25.000 linhas por arquivo.


In [8]:
# Simulação do particionamento de transações em 4 partes como mappers do Sqoop
mapper_dir = os.path.join(
    BASE, "raw/transactions/mapper_demo" # Define o diretório para armazenar as partes simuladas
)

if os.path.exists(mapper_dir):
    shutil.rmtree(mapper_dir) # Remove o diretório de mappers se existir

os.makedirs(mapper_dir) # Recria o diretório de mappers

conn = sqlite3.connect(DB_PATH)
tx = pd.read_sql("SELECT * FROM transactions", conn) # Lê todas as transações do banco de dados
conn.close()

parts = [tx.iloc[i::4].copy() for i in range(4)] # Divide as transações em 4 partes (simulando mappers)
mapper_files = []

for i, part in enumerate(parts):
    path = os.path.join(
        mapper_dir,
        f"part-m-{i:05d}.csv"
    )
    part.to_csv(path, index=False) # Salva cada parte como um arquivo CSV
    mapper_files.append(path)
    print(f"Mapper {i}: {len(part):,} linhas")

Mapper 0: 25,000 linhas
Mapper 1: 25,000 linhas
Mapper 2: 25,000 linhas
Mapper 3: 25,000 linhas


In [9]:
# Valida a totalidade e exclusividade dos IDs das transações particionadas
total = 0
all_ids = set() # Inicializa conjunto para armazenar IDs únicos de todas as partes

for path in mapper_files:
    part = pd.read_csv(path)
    total += len(part) # Soma o total de linhas de todas as partes
    all_ids.update(part["transaction_id"].tolist()) # Coleta todos os transaction_ids únicos

original_ids = set(tx["transaction_id"].tolist())

print(f"Total original: {len(tx):,}")
print(f"Total dividido: {total:,}")
print(f"IDs originais:  {len(original_ids):,}")
print(f"IDs nas partes: {len(all_ids):,}")

assert total == len(tx) # Compara o total de linhas com o original
assert all_ids == original_ids # Compara os conjuntos de IDs para exclusividade e totalidade

print("\n✓ Nenhum registro foi perdido.")

Total original: 100,000
Total dividido: 100,000
IDs originais:  100,000
IDs nas partes: 100,000

✓ Nenhum registro foi perdido.


In [10]:
# Verifica se existe sobreposição de transaction_id
id_sets = [
    set(pd.read_csv(path)["transaction_id"])
    for path in mapper_files # Cria conjuntos de IDs para cada parte
]

overlap = False

for i in range(4):
    for j in range(i + 1, 4):
        common = id_sets[i] & id_sets[j] # Verifica se há IDs comuns entre quaisquer duas partes
        if common:
            overlap = True
            print(f"✗ Sobreposição entre mapper {i} e {j}: {len(common)}")

assert not overlap # Garante que não houve sobreposição de IDs
print("✓ Nenhuma transação aparece em mais de uma parte.")

✓ Nenhuma transação aparece em mais de uma parte.


In [11]:
# Lista em iteração a estrutura de arquivos e diretórios final
for root, dirs, filenames in os.walk(BASE): # Percorre a estrutura de diretórios e arquivos a partir de BASE
    level = root.replace(BASE, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    for name in sorted(filenames):
        print(f"{indent}  └── {name}")

/
bronze/
gold/
raw/
  customers/
    └── customers_from_db.csv
    └── customers_synthetic.csv
  fraud_labels/
    └── fraud_labels.csv
  transactions/
    └── transactions_from_db.csv
    └── transactions_synthetic.csv
    mapper_demo/
      └── part-m-00000.csv
      └── part-m-00001.csv
      └── part-m-00002.csv
      └── part-m-00003.csv
silver/
_replica_demo/
  └── copia_1.csv
  └── copia_2.csv
  └── copia_3.csv


In [12]:
# Exibe amostras dos dados importados e das partes dos mappers
print("CUSTOMERS:")
display(c_check.head(5))

print("TRANSACTIONS:")
display(t_check.head(5))

print("PRIMEIRA PARTE / MAPPER:")
display(pd.read_csv(mapper_files[0]).head(5))

CUSTOMERS:


,customer_id,name,cpf,email,segment,credit_score,created_at
0,1,Ana Laura Campos,943.065.218-42,igor46@example.com,Premium,426,2026-07-03
1,2,Mariah Caldeira,586.237.094-38,jose48@example.com,High-Risk,481,2026-06-17
2,3,Kevin Cavalcante,530.629.814-15,theoda-costa@example.org,Standard,708,2025-11-06
3,4,Maria Laura Freitas,725.130.864-90,castrolucas@example.org,Premium,473,2026-02-07
4,5,Srta. Mirella Moura,570.814.239-14,emilly20@example.com,Premium,816,2025-10-04


TRANSACTIONS:


,transaction_id,customer_id,amount,transaction_type,timestamp,status,risk_score,is_fraud
0,9157,4627,156.683351,pagamento,2023-01-01,approved,89.859250,0
1,46882,9378,61.694980,compra,2023-01-01,approved,64.229229,0
2,91656,1385,41.844959,transferencia,2023-01-01,approved,0.183873,0
3,22248,8256,46.843159,pagamento,2023-01-01,declined,87.757724,0
4,15064,229,48.793758,compra,2023-01-01,approved,19.456056,0


PRIMEIRA PARTE / MAPPER:


,transaction_id,customer_id,amount,transaction_type,timestamp,status,risk_score,is_fraud
0,9157,4627,156.683351,pagamento,2023-01-01,approved,89.859250,0
1,15064,229,48.793758,compra,2023-01-01,approved,19.456056,0
2,91154,6640,67.946253,compra,2023-01-01,approved,33.266594,0
3,41645,7930,11.459087,compra,2023-01-01,approved,74.338931,0
4,37454,8447,109.035755,compra,2023-01-01,approved,49.587343,0


## ✅ Checkpoint

In [13]:
# Realiza um checkpoint automático para validar o progresso do Lab 2

# CHECKPOINT AUTOMÁTICO
db_ok = n_customers == 9993 and n_transactions == 100000
raw_ok = len(c_check) == 9993 and len(t_check) == 100000
parts_ok = total == 100000 and all_ids == original_ids and not overlap

print("=" * 65)
print("CHECKPOINT — LAB 2")
print("=" * 65)
print(f"[{'✓' if db_ok else '✗'}] Banco SQLite: 9.993 clientes / 100.000 transações")
print(f"[{'✓' if raw_ok else '✗'}] Dados exportados para raw")
print(f"[{'✓' if parts_ok else '✗'}] 4 partes simulando os mappers")
print("=" * 65)

if db_ok and raw_ok and parts_ok: # Verifica se as condições de sucesso para o laboratório foram atendidas
    print("🎉 LAB 2 CONCLUÍDO COM SUCESSO!")
else:
    print("⚠️ Revise as etapas marcadas com ✗.")

CHECKPOINT — LAB 2
[✓] Banco SQLite: 9.993 clientes / 100.000 transações
[✓] Dados exportados para raw
[✓] 4 partes simulando os mappers
🎉 LAB 2 CONCLUÍDO COM SUCESSO!
